In [ ]:
from pathlib import Path
import io
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import soundfile as sf
import matplotlib.pyplot as plt
from IPython.display import Audio, display

import os
ROOT = Path(os.environ.get("BTP_ROOT", Path.cwd())).resolve()
A_DIR = ROOT / "raw" / "A_narration" / "data"
files = sorted(A_DIR.glob("*/*.parquet"))
print(len(files), "parquet files found")   # expect 59

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 200)



In [ ]:
rows = []
for f in files:
    pf = pq.ParquetFile(f)
    rows.append({"split": f.parent.name, "file": f.name,
                 "rows": pf.metadata.num_rows, "row_groups": pf.metadata.num_row_groups})
file_table = pd.DataFrame(rows)
display(file_table.groupby("split").agg(files=("file", "count"), rows=("rows", "sum")))
display(file_table.head(30))

In [ ]:
text_cols = [c for c in pq.ParquetFile(files[0]).schema_arrow.names if c != "audio"]

parts = []
for f in files:
    part = pq.read_table(f, columns=text_cols).to_pandas()
    part.insert(0, "split", f.parent.name)       # which split folder it came from
    part.insert(1, "file", f.name)               # which parquet file
    part.insert(2, "row_in_file", range(len(part)))  # row number inside that file (used to find the audio)
    parts.append(part)

df = pd.concat(parts, ignore_index=True)
print(df.shape)   # expect (6718, 28)

In [ ]:
df.head(10)

In [ ]:
df.sample(10, random_state=0)

In [ ]:
print("Number of fields:", len(df.columns))

def is_missing(s):
    return s.astype(str).str.strip().isin(["N/A", "", "nan", "None"])

summary = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing (N/A or empty)": df.apply(is_missing).sum(),
    "unique values": df.nunique(),
})
summary

In [ ]:
T = "sentence-IPA-transcription"
df[["audio_id", "speakerId", "start_time", "end_time", T]].sample(15, random_state=1)

In [ ]:
# how many transcripts are missing?
missing = df[T].str.strip().isin(["N/A", ""])
print("missing transcripts:", missing.sum(), "of", len(df))
print(df.groupby("split")[T].apply(lambda s: s.str.strip().eq("N/A").sum()))

In [ ]:
# read the longest and the shortest transcripts
ok = df[~missing].copy()
ok["n_chars"] = ok[T].str.len()
print("Smallest Transcriptions:-")
display(ok.nsmallest(10, "n_chars")[["audio_id", T]])
print("Largest transcriptions:-")
display(ok.nlargest(5, "n_chars")[["audio_id", T]])

In [ ]:
print("Plot of Duration of each row/audio:-")
df["duration"] = df["end_time"] - df["start_time"]
display(df["duration"].describe())
df["duration"].hist(bins=60)
plt.xlabel("seconds"); plt.ylabel("clips"); plt.show()
print("total hours:", df["duration"].sum() / 3600)   # about 25.8

In [ ]:
_open_files = {}

def load_audio(row):
    """Return (waveform, sample_rate) for one row of df."""
    path = A_DIR / row["split"] / row["file"]
    if path not in _open_files:
        _open_files[path] = pq.ParquetFile(path)
    pf = _open_files[path]
    start = 0
    for g in range(pf.num_row_groups):
        n = pf.metadata.row_group(g).num_rows
        if row["row_in_file"] < start + n:
            tbl = pf.read_row_group(g, columns=["audio"])
            wav_bytes = tbl.column("audio")[row["row_in_file"] - start].as_py()["bytes"]
            return sf.read(io.BytesIO(wav_bytes))
        start += n
    raise IndexError("row not found")

In [ ]:
# read a transcript with waveform
row = df.sample(1, random_state=3).iloc[0]
y, sr = load_audio(row)

print("audio_id :", row["audio_id"])
print("speaker  :", row["speakerId"], "|", row["speaker1-gender"], row["speaker1-ageGroup"])
print("duration :", round(len(y) / sr, 2), "s   sample rate:", sr)
print("IPA      :", row[T])
display(Audio(y, rate=sr))

plt.figure(figsize=(10, 2))
plt.plot(np.arange(len(y)) / sr, y)
plt.xlabel("seconds"); plt.show()

In [ ]:
print("Listen to more audios, one after another:-")
sample = df[~missing].sample(10, random_state=7)
for _, row in sample.iterrows():
    y, sr = load_audio(row)
    print(f"{row['audio_id']}  |  {len(y)/sr:.1f}s  |  {row['speakerId']}")
    print(row[T])
    display(Audio(y, rate=sr))
    print("-" * 80)

In [ ]:
print("check that the audio matches the stated duration.\nIt prints only the clips whose real length differs from end_time − start_time by more than 0.05 s. On a random 30 you’ll usually see none")
chk = df.sample(30, random_state=5)
for _, row in chk.iterrows():
    y, sr = load_audio(row)
    diff = len(y) / sr - row["duration"]
    if abs(diff) > 0.05:
        print(row["audio_id"], "stated", round(row["duration"], 2), "actual", round(len(y) / sr, 2))
print("done")

In [ ]:
print("one row per publicSpeakerId")
spk = df.groupby("publicSpeakerId").agg(
    rows=("audio_id", "size"),
    hours=("duration", lambda x: round(x.sum() / 3600, 3)),
    n_speakerId=("speakerId", "nunique"),
    genders=("speaker1-gender", lambda x: " | ".join(sorted(x.unique()))),
    ages=("speaker1-ageGroup", lambda x: " | ".join(sorted(x.unique()))),
).sort_values("hours", ascending=False)

print("speakers:", len(spk))                       # expect 60
spk.head(15)

In [ ]:
print("Which speakers have conflicting information")
bad = spk[(spk["n_speakerId"] > 1)
          | spk["genders"].str.contains("|", regex=False)
          | spk["ages"].str.contains("|", regex=False)]
print("speakers with a conflict:", len(bad))      # expect 4
bad

In [ ]:
rev = df.groupby("speakerId")["publicSpeakerId"].nunique()
print("speakerIds that map to more than one publicSpeakerId:", (rev > 1).sum())   # expect 5
rev[rev > 1]

In [ ]:
print("Metadata for each conflicting grp")
conf_ids = rev[rev > 1].index
conf = df[df["speakerId"].isin(conf_ids)]
print("rows involved:", len(conf), "| hours:", round(conf["duration"].sum() / 3600, 3))

conf.groupby(["speakerId", "publicSpeakerId"]).agg(
    rows=("audio_id", "size"),
    hours=("duration", lambda x: round(x.sum() / 3600, 3)),
    gender=("speaker1-gender", lambda x: " | ".join(sorted(x.unique()))),
    age=("speaker1-ageGroup", lambda x: " | ".join(sorted(x.unique()))),
    place=("speaker1-placeOfRecording", lambda x: " | ".join(sorted(x.unique()))),
    spk2_gender=("speaker2-gender", lambda x: " | ".join(sorted(x.unique()))),
    splits=("split", lambda x: " | ".join(sorted(x.unique()))),
    sessions=("sentenceId", lambda x: x.str.split("_").str[0].nunique()),
)
print("play a few clips from each grp")
def play_group(speaker_id, public_id, n=3, seed=0, min_s=2, max_s=15):
    sub = df[(df["speakerId"] == speaker_id) & (df["publicSpeakerId"] == public_id)]
    sub = sub[(sub["duration"] >= min_s) & (sub["duration"] <= max_s) & (sub[T].str.strip() != "N/A")]
    print("=" * 90)
    print("speakerId       :", speaker_id)
    print("publicSpeakerId :", public_id, f"  ({len(sub)} clips of 2-15 s with a transcript)")
    for _, row in sub.sample(min(n, len(sub)), random_state=seed).iterrows():
        y, sr = load_audio(row)
        print("-" * 90)
        print(f"{row['audio_id']} | {len(y)/sr:.1f}s | {row['split']} | "
              f"{row['speaker1-gender']} {row['speaker1-ageGroup']} | place: {row['speaker1-placeOfRecording']} | "
              f"speaker2: {row['speaker2-gender']} {row['speaker2-ageGroup']}")
        print("domain:", row["Domain"], "| Q_Id:", row["Q_Id"])
        print("IPA   :", row[T])
        display(Audio(y, rate=sr))
pairs = (conf.groupby(["speakerId", "publicSpeakerId"]).size().reset_index()[["speakerId", "publicSpeakerId"]])
for _, p in pairs.iterrows():
    play_group(p["speakerId"], p["publicSpeakerId"], n=3)


tiny = conf[conf["speakerId"].str.startswith("[")]
print(len(tiny), "rows")
tiny[["audio_id", "speakerId", "publicSpeakerId", "duration", "speaker1-gender", "speaker2-gender", T]]
for _, row in tiny.iterrows():
    y, sr = load_audio(row)
    print(row["audio_id"], f"| {len(y)/sr:.1f}s |", row["publicSpeakerId"])
    print(row[T])
    display(Audio(y, rate=sr))


In [ ]:
multi = df[df["speakerId"].str.contains("and", regex=False) | df["speakerId"].str.startswith("[")]
multi.groupby(["speakerId", "publicSpeakerId"]).agg(
    rows=("audio_id", "size"),
    hours=("duration", lambda x: round(x.sum() / 3600, 3)))

In [ ]:
df["id_age_code"] = df["speakerId"].str.extract(r"(\d{3,4})_\d+")[0]
pd.crosstab(df["id_age_code"].fillna("none"), df["speaker1-ageGroup"])

In [ ]:
df["session"] = df["sentenceId"].str.split("_").str[0]
df["session_noslice"] = df["session"].str.replace(r"-slice\d+$", "", regex=True)

print("with slice number   :", df["session"].nunique())          # expect 2660
print("without slice number:", df["session_noslice"].nunique())  # expect 2333

# how many speakers per session?
df.groupby("session_noslice")["publicSpeakerId"].nunique().value_counts()

In [ ]:
s2 = df[df["speaker2-gender"] != "N/A"]
print("rows with speaker 2 info:", len(s2))                       # expect 11
s2[["speakerId", "publicSpeakerId", "speaker1-gender", "speaker2-gender", "speaker2-ageGroup"]].drop_duplicates()

In [ ]:
print("=" * 100)
print("CELL M1 - SPEAKER STATUS: which speakers can we trust, and which must stay in train only?")
print("=" * 100)

# One row per publicSpeakerId, with three facts about each:
#   n_speakerId : how many different raw speakerId strings point to this public ID
#   genders     : every distinct gender label found for this public ID, joined with " | "
#   ages        : every distinct age-group label found for this public ID, joined with " | "
spk = df.groupby("publicSpeakerId").agg(
    n_speakerId=("speakerId", "nunique"),
    genders=("speaker1-gender", lambda x: " | ".join(sorted(x.unique()))),
    ages=("speaker1-ageGroup", lambda x: " | ".join(sorted(x.unique()))),
)

# A public ID is "in conflict" if ANY of these is true:
#   - it is linked to more than one raw speakerId
#   - it has more than one gender label (the " | " separator appears in the text)
#   - it has more than one age-group label
conflict_ids = set(spk[(spk["n_speakerId"] > 1)
                       | spk["genders"].str.contains("|", regex=False)
                       | spk["ages"].str.contains("|", regex=False)].index)

# A "pair recording" is a row whose raw speakerId names two people:
#   - "X and Y" style IDs contain the text "and"
#   - the other style is a list written in square brackets, starting with "["
is_pair_recording = df["speakerId"].str.contains("and", regex=False) | df["speakerId"].str.startswith("[")

# Final label for every row, in priority order:
#   1) public ID is one of the conflicting ones -> "uncertain_identity"
#   2) otherwise, a pair recording               -> "multi_person"
#   3) everything else                           -> "ok"
status = np.where(df["publicSpeakerId"].isin(conflict_ids), "uncertain_identity",
         np.where(is_pair_recording, "multi_person", "ok"))

print(f"\nPublic speaker IDs found in the dataset: {len(spk)}   (expected 60)")
print(f"Public IDs flagged as conflicting      : {len(conflict_ids)}   (expected 4)")
print(f"Rows that belong to a pair recording    : {int(is_pair_recording.sum())}")
print("\nNumber of ROWS (audio clips) in each speaker_status class")
print("(expected: ok 6334, multi_person 208, uncertain_identity 176):\n")
print(pd.Series(status).value_counts().to_string())

In [ ]:
print("=" * 100)
print("CELL M2 - BUILD THE MASTER TABLE: one row per original audio clip, with our own clean columns")
print("=" * 100)

def clean(s):
    """Replace the dataset's text placeholders ('N/A', empty, 'nan', 'None') by a real empty value.
    Used ONLY for metadata columns; the transcript itself is never touched."""
    return s.where(~s.astype(str).str.strip().isin(["N/A", "", "nan", "None"]), None)

M = pd.DataFrame({
    # ---- identity of the clip ----
    "sample_id": ["A_" + str(i + 1).zfill(6) for i in range(len(df))],   # our own ID: A_000001 ... A_006718
    "source_dataset": "Narration",                                         # which dataset this row belongs to
    "original_audio_id": df["audio_id"],                                   # the dataset's own ID, unchanged
    "audio_path": "raw/A_narration/data/" + df["split"] + "/" + df["file"] + "#row=" + df["row_in_file"].astype(str),
                                                                           # where the audio lives: parquet file + row number
    "original_split": df["split"],                                         # train / validation / test as the authors had it
    # ---- timing ----
    "start_time": df["start_time"],                                        # segment start inside the longer recording (s)
    "end_time": df["end_time"],                                            # segment end inside the longer recording (s)
    "duration_seconds": (df["end_time"] - df["start_time"]).round(3),      # clip length, computed from the two above
    "duration_basis": "stated (end_time - start_time)",                    # reminder: not yet checked by decoding every file
    # ---- transcript ----
    "transcript_raw": df[T],                                               # the IPA text exactly as given - never edited
    "transcript_missing": df[T].str.strip().isin(["N/A", ""]),             # True when the transcript is a placeholder
    "transcript_normalized": None,                                         # empty for now; filled in Step 4
    # ---- speaker ----
    "speaker_id": df["publicSpeakerId"],                                   # the speaker key we use from now on
    "speaker_id_raw": df["speakerId"],                                     # the original raw speakerId, kept for reference
    "speaker_status": status,                                              # ok / uncertain_identity / multi_person (from M1)
    "is_pair_recording": is_pair_recording,                                # True when the raw ID names two people
    "train_only": status != "ok",                                          # True = this row may only go to the train split
    # ---- recording info ----
    "recording_session_id": df["sentenceId"].str.split("_").str[0].str.replace(r"-slice\d+$", "", regex=True),
                                                                           # sentenceId before "_", minus "-sliceNNNN" = one original recording
    "sentence_id": df["sentenceId"],
    "boundary_id": df["boundaryID"],
    "filename": df["filename"],
    # ---- prompt and speaker metadata (placeholders turned into empty values) ----
    "domain": clean(df["Domain"]),
    "elicitation_method": clean(df["Elicitation Method"]),
    "q_id": clean(df["Q_Id"]),
    "gender": clean(df["speaker1-gender"]),
    "age_group": clean(df["speaker1-ageGroup"]),
    "education_level": clean(df["speaker1-educationLevel"]),
    "place_of_recording": clean(df["speaker1-placeOfRecording"]),
    "type_of_place": clean(df["speaker1-typeOfPlace"]),
    "other_languages": clean(df["speaker1-otherLanguages-list"]),
    "speaker2_gender": clean(df["speaker2-gender"]),
    "speaker2_age_group": clean(df["speaker2-ageGroup"]),
    # ---- columns that later steps will fill in (empty for now) ----
    "quality_status": None, "exclusion_reason": None,
    "transcript_review_status": "not_reviewed",
    "language_status": None, "content_type": "narration",
    "duplicate_group_id": None, "final_split": None,
})

print(f"\nMaster table built: {M.shape[0]} rows x {M.shape[1]} columns   (expected 6718 x 39)")
print("\nFirst 5 rows of the master table (scroll right to see all columns):")
display(M.head())

In [ ]:
print("=" * 100)
print("CELL M3 - CHECKS: is the master table complete, unique and consistent with what we know?")
print("=" * 100)

# These two lines STOP the notebook with an error if something is wrong.
assert len(M) == 6718, "row count is not 6718"
assert M["sample_id"].is_unique and M["original_audio_id"].is_unique, "duplicate IDs found"
print("\n[OK] 6718 rows, every sample_id and every original_audio_id is unique.")
print(f"\nTotal duration of all clips: {round(M['duration_seconds'].sum() / 3600, 3)} hours   (expected 25.799)")

print("\n--- TABLE 1: size of each speaker_status class ---")
print("rows = clips, speakers = distinct speaker_id values, hours = total audio")
print("(expected: ok 52 speakers / 24.84 h; multi_person 4 / 0.44 h; uncertain_identity 4 / 0.52 h)")
display(M.groupby("speaker_status").agg(
    rows=("sample_id", "size"),
    speakers=("speaker_id", "nunique"),
    hours=("duration_seconds", lambda x: round(x.sum() / 3600, 3))))

print("\n--- TABLE 2: speaker_status by the authors' ORIGINAL split ---")
print("shows how many flagged clips the authors had put into test/validation (we will keep them out of there later)")
display(M.groupby(["original_split", "speaker_status"]).size().unstack(fill_value=0))

print("\n--- TABLE 3: which raw speakerId strings are flagged, and how many clips each ---")
display(M[M["speaker_status"] != "ok"].groupby(["speaker_status", "speaker_id_raw"]).size().rename("clips"))

In [ ]:
print("=" * 100)
print("CELL M4 - TRACE-BACK TEST: does each master row really point to the right raw audio/transcript?")
print("=" * 100)
print("We pick 5 random rows, open the raw parquet file named in audio_path, go to the stated row,")
print("and compare the audio_id and the transcript with what the master table says.\n")

chk = M.sample(5, random_state=11)
for _, r in chk.iterrows():
    path, row_n = r["audio_path"].split("#row=")                            # split "file#row=17" into file and row number
    t = pq.ParquetFile(ROOT / path).read(columns=[T, "audio_id"]).to_pandas().iloc[int(row_n)]   # read that row from the raw file
    print(r["sample_id"], "| audio_id match:", t["audio_id"] == r["original_audio_id"],
          "| transcript match:", t[T] == r["transcript_raw"])


In [ ]:
print("=" * 100)
print("CELL M5 - SAVE the master table to disk (two formats)")
print("=" * 100)

OUT = ROOT / "manifests"                      # folder for tables we create
OUT.mkdir(exist_ok=True)                      # create it if it does not exist yet
M.to_parquet(OUT / "master_A.parquet", index=False)                    # compact, exact; used by later code
M.to_csv(OUT / "master_A.csv", index=False, encoding="utf-8-sig")      # readable in Excel (utf-8-sig keeps the IPA symbols correct)

back = pd.read_parquet(OUT / "master_A.parquet")                       # read it back to prove the saved file is intact
print(f"\nSaved to: {OUT}")
print(f"Reloaded file shape: {back.shape}   (expected (6718, 39))")
print("Transcripts identical after saving and reloading:", back["transcript_raw"].equals(M["transcript_raw"]), "  (expected True)")

In [ ]:
print("=" * 100)
print("CELL M6 - DECISION LOG: write down every decision made in Step 3, so it can be cited in the thesis")
print("=" * 100)
import csv, datetime
today = datetime.date.today().isoformat()
st = M.groupby("speaker_status").agg(rows=("sample_id", "size"), hours=("duration_seconds", lambda x: round(x.sum() / 3600, 3)))

# each list = [date, step, decision, reason, rows affected, hours affected]
log = [
 [today, "3", "Narration speaker key = publicSpeakerId (60 speakers)", "More consistent than speakerId (62 values, pair recordings)", len(M), round(M["duration_seconds"].sum()/3600, 3)],
 [today, "3", "speaker_status=uncertain_identity for the 4 conflicting public IDs; train_only", "gender/age or several speakerIds per public ID", int(st.loc["uncertain_identity","rows"]), float(st.loc["uncertain_identity","hours"])],
 [today, "3", "speaker_status=multi_person for pair recordings ('X and Y'); train_only", "voice cannot be attributed to one speaker", int(st.loc["multi_person","rows"]), float(st.loc["multi_person","hours"])],
 [today, "3", "session id = sentenceId prefix without -sliceNNNN (2333 sessions)", "slice number is a piece of one recording", len(M), None],
 [today, "3", "duration = end_time - start_time (stated, not yet decoded)", "decoded check on 30 random clips matched; full check is Step 11", len(M), None],
]
with open(OUT / "decision_log_A.csv", "w", newline="", encoding="utf-8-sig") as f:
    w = csv.writer(f)
    w.writerow(["date", "step", "decision", "reason", "rows_affected", "hours_affected"])   # column headings
    w.writerows(log)                                                                          # one line per decision
print(f"\nDecision log saved to: {OUT / 'decision_log_A.csv'}")
print("Its content (one row per decision; add a new row every time you make a cleaning decision in later steps):")
display(pd.read_csv(OUT / "decision_log_A.csv"))

In [ ]:
print("=" * 100)
print("CELL M7 (optional) - SPEAKERS WHOSE NAME ALSO APPEARS IN A PAIR RECORDING")
print("=" * 100)
print("This does NOT change the master table. It only lists speakers marked 'ok' whose name stem")
print("(the name part of the raw speakerId) also appears in a pair recording. If the name means the same person,")
print("this person's voice would be in both train (pair recording) and, possibly, test (as a solo speaker) = leakage.\n")
import re

def stems(r):
    """Return the list of name stems inside a raw speakerId.
       '[..]' style  -> every 'name+digits_digits' piece found inside the brackets
       'XandY' style -> the two names split on 'and'
       normal style  -> the single name (trailing digits and timestamp removed)"""
    if r.startswith("["):
        return re.findall(r"([a-z]+)\d+_\d+", r)
    base = re.sub(r"\d+(_\d+)?$", "", r)
    return base.split("and") if "and" in r else [base]

# all name stems that appear in any pair recording
pair_stems = {s for r in M.loc[M["is_pair_recording"], "speaker_id_raw"].unique() for s in stems(r)}
print("Name stems found in pair recordings:", sorted(pair_stems))

# solo rows (not pair recordings) whose single name stem is one of those stems and who are marked ok
solo = M[~M["is_pair_recording"]].copy()
solo["name_stem"] = solo["speaker_id_raw"].map(lambda r: stems(r)[0])
linked = solo[solo["name_stem"].isin(pair_stems) & (solo["speaker_status"] == "ok")]

print("\nSolo speakers marked 'ok' that share a name with a pair-recording participant (expected 3: jorampaul, joramnikam, joramchobin):")
display(linked.groupby(["speaker_id", "speaker_id_raw", "name_stem"]).agg(
    rows=("sample_id", "size"), hours=("duration_seconds", lambda x: round(x.sum() / 3600, 3))))


In [ ]:
# --- load_audio: reads ONE clip's waveform out of the raw parquet file (same function as before) ---
_open_files = {}                                   # remember opened parquet files so we do not reopen them every time
def load_audio(row):
    path = A_DIR / row["split"] / row["file"]      # which parquet file holds this clip
    if path not in _open_files:
        _open_files[path] = pq.ParquetFile(path)
    pf = _open_files[path]
    start = 0
    for g in range(pf.num_row_groups):             # parquet files are cut into blocks ("row groups"); find the block holding our row
        n = pf.metadata.row_group(g).num_rows
        if row["row_in_file"] < start + n:
            tbl = pf.read_row_group(g, columns=["audio"])
            wav_bytes = tbl.column("audio")[row["row_in_file"] - start].as_py()["bytes"]
            return sf.read(io.BytesIO(wav_bytes))  # decode the WAV bytes into (waveform, sample rate)
        start += n

# --- play_clips: plays a few random clips from a group of rows, with the information you need to judge them ---
#   mask  : True/False per row of M, saying which rows belong to the group
#   n     : how many clips to play
#   min_s / max_s : only clips of this length (very short ones are hard to judge by ear)
def play_clips(mask, title, n=4, seed=0, min_s=3, max_s=15):
    sub = M[mask & ~M["transcript_missing"] & (M["duration_seconds"] >= min_s) & (M["duration_seconds"] <= max_s)]
    print("=" * 100)
    print(title)
    print(f"{int(mask.sum())} clips in this group in total; {len(sub)} of them are {min_s}-{max_s} s long and have a transcript; playing {min(n, len(sub))}")
    print("=" * 100)
    for i, r in sub.sample(min(n, len(sub)), random_state=seed).iterrows():
        y, sr = load_audio(df.loc[i])              # df and M have the same row order, so index i finds the same clip in df
        print("-" * 100)
        print(f"{r['sample_id']} | {len(y)/sr:.1f}s | raw speakerId: {r['speaker_id_raw']}")
        print(f"gender/age: {r['gender']} {r['age_group']} | speaker2: {r['speaker2_gender']} {r['speaker2_age_group']} | place: {r['place_of_recording']} | session: {r['recording_session_id']}")
        print("IPA:", r["transcript_raw"])
        display(Audio(y, rate=sr))

In [ ]:
from IPython.display import Audio, display
import soundfile as sf

print("METADATA of the speakers compared below")
print("solo = a normal single-speaker recording; PAIR = a recording where two people talk")
print("(gender/age are empty for pair recordings: the dataset did not record them)\n")

# each entry = a True/False mask over the rows of M
grp = {
  "solo  jorampaul":              M["speaker_id_raw"].str.startswith("jorampaul1830"),
  "PAIR  jorampaul+joramyama":    M["speaker_id_raw"].str.startswith("jorampaulandjoramyama"),
  "solo  joramnikam":             M["speaker_id_raw"].str.startswith("joramnikam1830"),
  "PAIR  joramnikam+joramchobin": M["speaker_id_raw"].str.startswith("joramnikamandjoramchobin"),
  "solo  joramchobin":            M["speaker_id_raw"].str.startswith("joramchobin4560"),
}

rows_ = []
for name, m in grp.items():
    s = M[m]
    rows_.append(dict(
        group=name,
        clips=len(s),
        hours=round(s["duration_seconds"].sum() / 3600, 3),
        gender=" | ".join(sorted(s["gender"].dropna().unique())),
        age=" | ".join(sorted(s["age_group"].dropna().unique())),
        place=" | ".join(sorted(s["place_of_recording"].dropna().unique())),
        sessions=s["recording_session_id"].nunique(),          # how many separate recordings this group covers
        first_session=s["recording_session_id"].min()))        # the recording's ID (it starts with the recording date)
display(pd.DataFrame(rows_).set_index("group"))

In [ ]:
play_clips(grp["solo  jorampaul"], "1a) SOLO speaker: jorampaul (raw ID jorampaul1830_...)", n=3)
play_clips(grp["PAIR  jorampaul+joramyama"], "1b) PAIR recording: jorampaul AND joramyama", n=4)

In [ ]:
play_clips(grp["solo  joramnikam"], "2a) SOLO speaker: joramnikam (raw ID joramnikam1830_...)", n=3)
play_clips(grp["PAIR  joramnikam+joramchobin"], "2b) PAIR recording: joramnikam AND joramchobin", n=4)

In [ ]:
play_clips(grp["solo  joramchobin"], "3a) SOLO 'joramchobin' (this is only 1 clip)", n=1, min_s=0)
play_clips(grp["PAIR  joramnikam+joramchobin"], "3b) the pair recording again, different clips", n=4, seed=5)

In [ ]:
print("=" * 100)
print("CELL L6 - MARK THE 3 NAME-LINKED SPEAKERS AS 'linked_to_pair' (train only)")
print("=" * 100)

# 'linked' comes from Cell M7: solo speakers marked ok whose name also appears in a pair recording
linked_ids = set(linked["speaker_id"])
print("speaker_id values being re-labelled:", len(linked_ids), "(expected 3)")

# only change rows that are currently "ok"; the flagged classes from M1 stay exactly as they are
mask = M["speaker_id"].isin(linked_ids) & (M["speaker_status"] == "ok")
before = M["speaker_status"].value_counts().to_dict()

M.loc[mask, "speaker_status"] = "linked_to_pair"      # new status
M.loc[mask, "train_only"] = True                      # these rows may only go to the train split

print("\nrows re-labelled:", int(mask.sum()), "| hours:", round(M.loc[mask, "duration_seconds"].sum() / 3600, 3), "(expected 622 rows, about 1.864 h)")
print("\nspeaker_status BEFORE:", before)
print("speaker_status AFTER :", M["speaker_status"].value_counts().to_dict())
print("\nFinal table of speaker_status classes (rows, speakers, hours, and whether they are train-only):")
display(M.groupby("speaker_status").agg(
    rows=("sample_id", "size"),
    speakers=("speaker_id", "nunique"),
    hours=("duration_seconds", lambda x: round(x.sum() / 3600, 3)),
    train_only=("train_only", "all")))

# ---- save the updated master table and add the decision to the log ----
M.to_parquet(OUT / "master_A.parquet", index=False)
M.to_csv(OUT / "master_A.csv", index=False, encoding="utf-8-sig")
new_row = [today, "3", "speaker_status=linked_to_pair (train_only) for solo speakers whose name appears in a pair recording: jorampaul, joramnikam, joramchobin",
           "same person cannot be excluded; not verified by listening (decided to be safe)", int(mask.sum()), round(float(M.loc[mask, "duration_seconds"].sum() / 3600), 3)]
with open(OUT / "decision_log_A.csv", "a", newline="", encoding="utf-8-sig") as f:
    csv.writer(f).writerow(new_row)
print("\nSaved master_A.parquet / master_A.csv again and added one line to decision_log_A.csv. Last log lines:")
display(pd.read_csv(OUT / "decision_log_A.csv").tail(2))

### Drawing PLOTS for understanding

In [ ]:
import matplotlib.pyplot as plt

plt.rcParams.update({"figure.dpi": 100, "axes.spines.top": False, "axes.spines.right": False, "font.size": 10})
M["hours"] = M["duration_seconds"] / 3600                      # clip length in hours (handy for every plot)

STATUS_ORDER = ["ok", "linked_to_pair", "uncertain_identity", "multi_person"]
STATUS_COLOR = {"ok": "#4C9F70", "linked_to_pair": "#E0A030", "uncertain_identity": "#D1603D", "multi_person": "#7B6CC4"}

def label_of(s):
    """One label per speaker: the single value if there is one, 'unknown' if empty, 'conflicting' if several."""
    u = sorted(s.dropna().unique())
    return "unknown" if len(u) == 0 else (u[0] if len(u) == 1 else "conflicting")

# one row per speaker_id: how many clips and hours, plus gender / age / status of that speaker
spk_tbl = M.groupby("speaker_id").agg(
    rows=("sample_id", "size"),
    hours=("hours", "sum"),
    gender=("gender", label_of),
    age=("age_group", label_of),
    status=("speaker_status", "first"),
).reset_index()
print("Speaker table built:", len(spk_tbl), "speakers (expected 60)")
display(spk_tbl.sort_values("hours", ascending=False).head(10).round(3))

In [ ]:
print("=" * 100)
print("PLOT 1 - SPEAKER STATUS: how much of the data comes from trustworthy speakers?")
print("=" * 100)
print("""
What each speaker_status means:
  ok                  One identifiable person, consistent gender/age labels, a normal single-speaker recording.
                      -> may be placed in train, validation OR test.
  linked_to_pair      A normal solo speaker, BUT their name also appears in a two-person recording, so they
                      might be the same person as someone in that pair. -> train only (to avoid leaking a voice into test).
  uncertain_identity  The dataset's public speaker ID is inconsistent (several raw IDs, or two different genders / age
                      groups for one ID), so we cannot be sure whose voice it is. -> train only.
  multi_person        A recording where two named people talk ('X and Y'). The voice cannot be assigned to one
                      person. -> train only.
""")
g = M.groupby("speaker_status").agg(clips=("sample_id", "size"), hours=("hours", "sum"), speakers=("speaker_id", "nunique")).reindex(STATUS_ORDER)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col, title in zip(axes, ["clips", "hours", "speakers"], ["Number of clips (rows)", "Hours of audio", "Number of speakers"]):
    bars = ax.bar(g.index, g[col], color=[STATUS_COLOR[s] for s in g.index])
    ax.bar_label(bars, fmt="%.1f" if col == "hours" else "%d")
    ax.set_title(title); ax.tick_params(axis="x", rotation=25)
fig.suptitle("Plot 1: speaker_status of the Narration dataset (6,718 clips)", y=1.02)
plt.tight_layout(); plt.show()
print("Share of total hours by class (%):")
print((100 * g["hours"] / g["hours"].sum()).round(1).to_string())

In [ ]:
print("=" * 100)
print("PLOT 2 - GENDER OF SPEAKERS (number of speakers and hours, coloured by speaker_status)")
print("=" * 100)
print("'unknown' = no gender recorded (all pair recordings); 'conflicting' = the same speaker has two different gender labels.\n")
GENDER_ORDER = ["Female", "Male", "unknown", "conflicting"]
cnt = spk_tbl.pivot_table(index="gender", columns="status", values="hours", aggfunc="count", fill_value=0).reindex(index=GENDER_ORDER, columns=STATUS_ORDER, fill_value=0)
hrs = spk_tbl.pivot_table(index="gender", columns="status", values="hours", aggfunc="sum", fill_value=0).reindex(index=GENDER_ORDER, columns=STATUS_ORDER, fill_value=0)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
cnt.plot(kind="bar", stacked=True, ax=axes[0], color=[STATUS_COLOR[s] for s in STATUS_ORDER], legend=False)
axes[0].set_title("Number of speakers"); axes[0].set_xlabel(""); axes[0].tick_params(axis="x", rotation=0)
hrs.plot(kind="bar", stacked=True, ax=axes[1], color=[STATUS_COLOR[s] for s in STATUS_ORDER])
axes[1].set_title("Hours of audio"); axes[1].set_xlabel(""); axes[1].tick_params(axis="x", rotation=0)
axes[1].legend(title="speaker_status", frameon=False)
fig.suptitle("Plot 2: gender", y=1.02); plt.tight_layout(); plt.show()
print("Speakers per gender:\n", cnt.sum(axis=1).to_string(), "\n\nHours per gender:\n", hrs.sum(axis=1).round(2).to_string())

In [ ]:
print("=" * 100)
print("PLOT 3 - AGE GROUP OF SPEAKERS (number of speakers and hours, coloured by speaker_status)")
print("=" * 100)
AGE_ORDER = ["18-30", "30-45", "45-60", "60+", "unknown", "conflicting"]
cnt = spk_tbl.pivot_table(index="age", columns="status", values="hours", aggfunc="count", fill_value=0).reindex(index=AGE_ORDER, columns=STATUS_ORDER, fill_value=0)
hrs = spk_tbl.pivot_table(index="age", columns="status", values="hours", aggfunc="sum", fill_value=0).reindex(index=AGE_ORDER, columns=STATUS_ORDER, fill_value=0)
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
cnt.plot(kind="bar", stacked=True, ax=axes[0], color=[STATUS_COLOR[s] for s in STATUS_ORDER], legend=False)
axes[0].set_title("Number of speakers"); axes[0].set_xlabel(""); axes[0].tick_params(axis="x", rotation=0)
hrs.plot(kind="bar", stacked=True, ax=axes[1], color=[STATUS_COLOR[s] for s in STATUS_ORDER])
axes[1].set_title("Hours of audio"); axes[1].set_xlabel(""); axes[1].tick_params(axis="x", rotation=0)
axes[1].legend(title="speaker_status", frameon=False)
fig.suptitle("Plot 3: age group", y=1.02); plt.tight_layout(); plt.show()
print("Speakers per age group:\n", cnt.sum(axis=1).to_string(), "\n\nHours per age group:\n", hrs.sum(axis=1).round(2).to_string())

In [ ]:
print("=" * 100)
print("PLOT 4 - GENDER x AGE: hours of audio in each combination (heat map)")
print("=" * 100)
print("Empty or dark cells show groups with very little data - the model will hear few such voices.\n")
hm = spk_tbl.pivot_table(index="gender", columns="age", values="hours", aggfunc="sum", fill_value=0).reindex(index=GENDER_ORDER, columns=AGE_ORDER, fill_value=0)
fig, ax = plt.subplots(figsize=(8, 3.6))
im = ax.imshow(hm.values, cmap="Blues")
ax.set_xticks(range(len(hm.columns)), hm.columns); ax.set_yticks(range(len(hm.index)), hm.index)
for i in range(hm.shape[0]):
    for j in range(hm.shape[1]):
        ax.text(j, i, f"{hm.values[i, j]:.1f}", ha="center", va="center", color="white" if hm.values[i, j] > hm.values.max() * 0.5 else "black")
fig.colorbar(im, label="hours"); ax.set_title("Plot 4: hours by gender and age group"); plt.tight_layout(); plt.show()
sp_hm = spk_tbl.pivot_table(index="gender", columns="age", values="hours", aggfunc="count", fill_value=0).reindex(index=GENDER_ORDER, columns=AGE_ORDER, fill_value=0)
print("Number of speakers in each combination:"); display(sp_hm)

In [ ]:
print("=" * 100)
print("PLOT 5 - HOURS PER SPEAKER: is the dataset dominated by a few voices?")
print("=" * 100)
s = spk_tbl.sort_values("hours", ascending=False).reset_index(drop=True)
fig, axes = plt.subplots(1, 2, figsize=(14, 4.2), gridspec_kw={"width_ratios": [2, 1]})
axes[0].bar(range(len(s)), s["hours"], color=[STATUS_COLOR[x] for x in s["status"]])
axes[0].set_title("Hours of audio per speaker (sorted; colour = speaker_status)"); axes[0].set_xlabel("speaker rank"); axes[0].set_ylabel("hours")
axes[0].legend(handles=[plt.Rectangle((0, 0), 1, 1, color=STATUS_COLOR[k]) for k in STATUS_ORDER], labels=STATUS_ORDER, frameon=False)
cum = 100 * s["hours"].cumsum() / s["hours"].sum()
axes[1].plot(range(1, len(s) + 1), cum, marker="o", ms=3)
axes[1].axhline(50, ls=":", c="grey"); axes[1].set_title("Cumulative share of all hours (%)"); axes[1].set_xlabel("number of top speakers"); axes[1].set_ylabel("% of total hours")
plt.tight_layout(); plt.show()
print(f"Top 1 speaker  : {cum.iloc[0]:.1f}% of all hours")
print(f"Top 3 speakers : {cum.iloc[2]:.1f}%")
print(f"Top 5 speakers : {cum.iloc[4]:.1f}%")
print(f"Top 10 speakers: {cum.iloc[9]:.1f}%")
print(f"Speakers with less than 0.1 h of audio: {(s['hours'] < 0.1).sum()} of {len(s)}")

In [ ]:
print("=" * 100)
print("PLOT 6 - CLIP DURATIONS: how long are the clips, and where is the audio?")
print("=" * 100)
bins = [0, 0.5, 2, 5, 10, 20, 30, 60, 200]
labels = ["<0.5 s", "0.5-2 s", "2-5 s", "5-10 s", "10-20 s", "20-30 s", "30-60 s", ">60 s"]
M["dur_bin"] = pd.cut(M["duration_seconds"], bins=bins, labels=labels, right=False)
db = M.groupby("dur_bin", observed=False).agg(clips=("sample_id", "size"), hours=("hours", "sum"))
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].hist(M["duration_seconds"].clip(upper=60), bins=60, color="#4C78A8")
axes[0].set_title("Duration of each clip (values above 60 s are piled into the last bar)"); axes[0].set_xlabel("seconds"); axes[0].set_ylabel("number of clips")
b = axes[1].bar(db.index.astype(str), db["clips"], color="#4C78A8"); axes[1].bar_label(b); axes[1].set_title("Clips per duration range"); axes[1].tick_params(axis="x", rotation=40)
b = axes[2].bar(db.index.astype(str), db["hours"], color="#E0A030"); axes[2].bar_label(b, fmt="%.1f"); axes[2].set_title("Hours of audio per duration range"); axes[2].tick_params(axis="x", rotation=40)
plt.tight_layout(); plt.show()
print(f"Clips longer than 30 s: {(M['duration_seconds'] > 30).sum()} clips, {M.loc[M['duration_seconds'] > 30, 'hours'].sum():.2f} h "
      f"({100 * M.loc[M['duration_seconds'] > 30, 'hours'].sum() / M['hours'].sum():.1f}% of all audio) - these cannot be used by models limited to 30 s windows.")
print(f"Clips shorter than 0.5 s: {(M['duration_seconds'] < 0.5).sum()} clips")

In [ ]:
print("=" * 100)
print("PLOT 7 - MISSING TRANSCRIPTS: how much audio has no text, per original split and per speaker_status?")
print("=" * 100)
M["transcript_state"] = np.where(M["transcript_missing"], "missing (N/A)", "has transcript")
by_split = M.groupby(["original_split", "transcript_state"]).agg(clips=("sample_id", "size"), hours=("hours", "sum")).reset_index()
by_stat = M.groupby(["speaker_status", "transcript_state"]).agg(clips=("sample_id", "size"), hours=("hours", "sum")).reset_index()
fig, axes = plt.subplots(2, 2, figsize=(13, 7))
for r, (tbl, idx) in enumerate([(by_split, "original_split"), (by_stat, "speaker_status")]):
    for c, val in enumerate(["clips", "hours"]):
        pv = tbl.pivot(index=idx, columns="transcript_state", values=val).fillna(0)
        if idx == "speaker_status": pv = pv.reindex(STATUS_ORDER)
        pv.plot(kind="bar", stacked=True, ax=axes[r, c], color=["#4C9F70", "#D1603D"], legend=(r == 0 and c == 1))
        axes[r, c].set_title(f"{val} by {idx}"); axes[r, c].set_xlabel(""); axes[r, c].tick_params(axis="x", rotation=0 if r == 0 else 20)
plt.suptitle("Plot 7: clips with and without a transcript", y=1.01); plt.tight_layout(); plt.show()
miss = M[M["transcript_missing"]]
print(f"Missing transcripts: {len(miss)} clips = {miss['hours'].sum():.2f} h = {100 * miss['hours'].sum() / M['hours'].sum():.1f}% of all audio")
print("They appear in every original split (including test and validation):", miss.groupby("original_split").size().to_dict())

In [ ]:
print("=" * 100)
print("PLOT 8 - ORIGINAL SPLITS (the authors' train / validation / test): sizes and speaker mix")
print("=" * 100)
print("We will NOT use these splits as they are (speakers appear in several of them), but this shows what the authors made.\n")
sp = M.groupby(["original_split", "speaker_status"]).agg(clips=("sample_id", "size"), hours=("hours", "sum")).reset_index()
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, val in zip(axes, ["clips", "hours"]):
    pv = sp.pivot(index="original_split", columns="speaker_status", values=val).reindex(columns=STATUS_ORDER).fillna(0).reindex(["train", "validation", "test"])
    pv.plot(kind="bar", stacked=True, ax=ax, color=[STATUS_COLOR[s] for s in STATUS_ORDER], legend=(val == "hours"))
    ax.set_title(val); ax.set_xlabel(""); ax.tick_params(axis="x", rotation=0)
plt.tight_layout(); plt.show()
ov = M.groupby("speaker_id")["original_split"].nunique()
print(f"Speakers that appear in more than one of the authors' splits: {(ov > 1).sum()} of {len(ov)}")
print(f"Speakers that appear in all three: {(ov == 3).sum()}")

In [ ]:
print("=" * 100)
print("PLOT 9 - WHAT PEOPLE TALK ABOUT (domain) AND WHERE THEY WERE RECORDED (place)")
print("=" * 100)
print("'(no label)' = the dataset has no domain / place for these clips.\n")
dom = M.assign(domain=M["domain"].fillna("(no label)")).groupby("domain").agg(clips=("sample_id", "size"), hours=("hours", "sum")).sort_values("hours")
plc = M.assign(place=M["place_of_recording"].fillna("(no label)")).groupby("place").agg(clips=("sample_id", "size"), hours=("hours", "sum")).sort_values("hours")
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
b = axes[0].barh(dom.index, dom["hours"], color="#4C78A8"); axes[0].bar_label(b, fmt="%.1f"); axes[0].set_title("Hours per domain")
b = axes[1].barh(plc.index, plc["hours"], color="#72B7B2"); axes[1].bar_label(b, fmt="%.1f"); axes[1].set_title("Hours per place of recording")
plt.tight_layout(); plt.show()
print(f"Clips without any domain label: {M['domain'].isna().sum()} ({M.loc[M['domain'].isna(), 'hours'].sum():.2f} h)")
print(f"Clips without a place label   : {M['place_of_recording'].isna().sum()} ({M.loc[M['place_of_recording'].isna(), 'hours'].sum():.2f} h)")

In [ ]:
print("=" * 100)
print("PLOT 10 - SPEAKING RATE AND TRANSCRIPT LENGTH (finds mismatched transcripts)")
print("=" * 100)
print("Characters per second = number of characters (without spaces) / clip length. Very low = text too short for the audio;")
print("very high = text too long for the audio. Both can mean a wrong transcript or a wrong cut. Only clips >= 0.5 s with a transcript are used.\n")
tx = M[~M["transcript_missing"] & (M["duration_seconds"] >= 0.5)].copy()
tx["n_chars"] = tx["transcript_raw"].str.replace(" ", "", regex=False).str.len()
tx["n_words"] = tx["transcript_raw"].str.split().str.len()
tx["cps"] = tx["n_chars"] / tx["duration_seconds"]
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].hist(tx["cps"].clip(upper=30), bins=60, color="#4C78A8"); axes[0].axvline(tx["cps"].median(), c="red", ls="--", label=f"median {tx['cps'].median():.1f}")
axes[0].set_title("Characters per second"); axes[0].set_xlabel("chars / s (above 30 piled into the last bar)"); axes[0].legend(frameon=False)
axes[1].scatter(tx["duration_seconds"].clip(upper=60), tx["n_chars"].clip(upper=1500), s=3, alpha=0.3, c="#4C78A8")
axes[1].set_title("Transcript length vs clip length"); axes[1].set_xlabel("clip duration (s, capped at 60)"); axes[1].set_ylabel("characters (capped at 1500)")
axes[2].hist(tx["n_words"].clip(upper=100), bins=50, color="#E0A030"); axes[2].set_title("Words per transcript"); axes[2].set_xlabel("words (above 100 piled into the last bar)")
plt.tight_layout(); plt.show()
print(tx["cps"].describe().round(2).to_string())
print(f"\nClips with fewer than 3 characters/second: {(tx['cps'] < 3).sum()}   |   more than 20: {(tx['cps'] > 20).sum()}")

In [ ]:
print("=" * 100)
print("PLOT 11 - WHEN WAS THE DATA RECORDED? (hours per month, from the recording session IDs)")
print("=" * 100)
print("The session ID starts with the recording date. This shows recording campaigns - clips from different campaigns may sound different.\n")
M["rec_month"] = pd.to_datetime(M["recording_session_id"].str[1:9], format="%Y%m%d", errors="coerce").dt.to_period("M").astype(str)
tm = M.groupby("rec_month").agg(clips=("sample_id", "size"), hours=("hours", "sum"), sessions=("recording_session_id", "nunique"))
fig, ax = plt.subplots(figsize=(11, 3.8))
b = ax.bar(tm.index, tm["hours"], color="#4C78A8"); ax.bar_label(b, fmt="%.1f"); ax.set_title("Hours of audio recorded per month"); ax.tick_params(axis="x", rotation=45)
plt.tight_layout(); plt.show()
display(tm.round(2))

In [ ]:
log = pd.read_csv(OUT / "decision_log_A.csv")
log = log.drop(index=len(log) - 1)                     # drop the last (duplicate) line
log.to_csv(OUT / "decision_log_A.csv", index=False, encoding="utf-8-sig")
log[["step", "decision", "rows_affected", "hours_affected"]]